# Audit of the fitting notebook

This notebook checks whether the models in `fitting.ipynb` are safe to put in the report. It reruns everything it needs, so it can be read on its own. Each check ends with a verdict in bold. The summary at the end lists what each check did, what it found and which section of the report it belongs in.

**Note.** On 1 October 2026, `fitting.ipynb` was replaced by `1_two_experiments.ipynb` and `2_all_runs.ipynb`. Those notebooks use the report's kernel statistics from `report_kernel.py`. This audit still checks the earlier version, which is in git at commit 50ffc43. Every mention of `fitting.ipynb` below means that version.

The eight checks are these.

1. Section 1 writes out the two formulas, the accelerated failure time model and the logistic model.
2. Section 2 tests for lookahead, which would mean that the inputs already contain the answer.
3. Section 3 measures how much the three kernel numbers overlap with each other.
4. Section 4 counts whether there are enough grokked runs for the number of coefficients.
5. Section 5 asks whether weight decay has a constant effect, and whether it is fair to train on one level and test on another.
6. Section 6 asks whether a large L1 model over every logged number is worth adding.
7. Section 7 fits one kernel number at a time to see which one is most useful.
8. Section 8 tests interaction terms between the kernel numbers.

The section and equation numbers of the report refer to the draft of 1 October 2026. The older `docs/report.tex` in the repository does not have those sections yet.

## Setup

This section loads the same data with the same definitions as Parts 5 to 8 of `fitting.ipynb`. The grokking step `T` is the first checkpoint at which the test accuracy reaches 0.95. A run that never gets there is censored at step 200,000. The kernel numbers are read at step 2,000. Weight decay 3e-3 is left out because none of its runs memorise.

When we load the numbers at step 2,000 we rename `S_c` and `R_c` to `S_t` and `R_t`, the names the report uses. The file also has uncentred columns called `S_t` and `R_t`. We rename those to `S_uncentred` and `R_uncentred`, so the two versions cannot be mixed up.

The next cell builds the three sets `wd`, `no_wd` and `pooled`, and prints the first rows of the pooled set. Each row is one run, with its cell, its settings, its target and its three kernel numbers.

In [1]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor
from lifelines import CoxPHFitter, LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
LANDMARK, BUDGET = 2_000, 200_000
kernel = ['A_t', 'S_t', 'R_t']

runs = pd.read_parquet('data/runs.parquet')
acc = pd.read_parquet('data/checkpoints.parquet', columns=['run_id', 'step', 'test_acc'])

events = runs[['run_id', 'alpha', 'width', 'eta_kappa', 'seed', 'last_step']].set_index('run_id')
events['t_gen95'] = acc[acc['test_acc'] >= 0.95].groupby('run_id')['step'].min()
events = events.reset_index()
events['grokked'] = events['t_gen95'].notna().astype(int)
events['T'] = events['t_gen95'].fillna(events['last_step'])
events['cell'] = (events['alpha'].astype(str) + '_' + events['width'].astype(str) + '_'
                  + events['eta_kappa'].map('{:g}'.format))

# every number logged at step 2,000; Section 6 uses all of them
at_landmark = pd.read_parquet('data/checkpoints.parquet', filters=[('step', '==', LANDMARK)])
at_landmark = at_landmark.drop(columns=['alpha', 'width', 'eta_kappa', 'seed'])
at_landmark = at_landmark.rename(columns={'S_t': 'S_uncentred', 'R_t': 'R_uncentred', 'S_c': 'S_t', 'R_c': 'R_t'})

data = events.merge(at_landmark, on='run_id')
data = data[data['eta_kappa'] <= 1e-3].reset_index(drop=True)

POOLED = 'all levels up to 1e-3'
wd = data[np.isclose(data['eta_kappa'], 3e-5)].copy()
no_wd = data[data['eta_kappa'] == 0].copy()
pooled = data.copy()
sets = {'weight decay 3e-5': wd, 'no weight decay': no_wd, POOLED: pooled}

pooled[['run_id', 'cell', 'alpha', 'width', 'eta_kappa', 'T', 'grokked'] + kernel].head()

,run_id,cell,alpha,width,eta_kappa,T,grokked,A_t,S_t,R_t
0,ntk_N50_a0.5_wd0_s0,0.5_50_0,0.5,50,0.0,4981.0,1,0.113876,1.318714,0.121258
1,ntk_N50_a0.5_wd0_s1,0.5_50_0,0.5,50,0.0,8319.0,1,0.106563,1.348164,0.120257
2,ntk_N50_a0.5_wd0_s2,0.5_50_0,0.5,50,0.0,6776.0,1,0.109269,1.354164,0.104921
3,ntk_N50_a0.5_wd0_s3,0.5_50_0,0.5,50,0.0,8000.0,1,0.112150,1.314184,0.115434
4,ntk_N50_a0.5_wd0_s4,0.5_50_0,0.5,50,0.0,6776.0,1,0.113564,1.413058,0.117339


Two small helpers save us from copying the same lines twenty times.

- `settings_dummies(df)` makes the 0/1 columns for `alpha`, width and weight decay, as in Part 7 of `fitting.ipynb`. The smallest value of each setting is the baseline and gets no column.
- `hold_out_each_cell(df, X, scale, model)` is the hold-out loop of Parts 7 and 8 of `fitting.ipynb`. For each cell it fits a model on every other cell and predicts the runs of that cell. The columns listed in `scale` are standardised with the training runs each time. It returns the predicted median grokking step when `model='aft'`, or the probability of grokking when `model='logistic'`.

The cell ends by printing the 0/1 columns next to the kernel numbers for the first five runs.

In [2]:
def settings_dummies(df):
    """0/1 columns for alpha, width and weight decay. The smallest value of each is the baseline and gets no column."""
    return pd.get_dummies(df[['alpha', 'width', 'eta_kappa']].astype('category'), drop_first=True, dtype=float)


def hold_out_each_cell(df, X, scale, model='aft'):
    """For each cell: fit on every other cell, predict this one. Returns the predicted median
    grokking step (model='aft') or the probability of grokking (model='logistic') for every run."""
    out = pd.Series(index=df.index, dtype=float)
    for cell in df['cell'].unique():
        train = df['cell'] != cell
        test = df['cell'] == cell
        X_fold = X.copy()
        X_fold[scale] = (X[scale] - X.loc[train, scale].mean()) / X.loc[train, scale].std()
        if model == 'aft':
            fit = LogNormalAFTFitter(penalizer=0.01).fit(X_fold[train].assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
            out[test] = fit.predict_median(X_fold[test]).to_numpy().ravel()
        else:
            fit = LogisticRegression(max_iter=1000).fit(X_fold[train], df.loc[train, 'grokked'])
            out[test] = fit.predict_proba(X_fold[test])[:, 1]
    return out


settings_dummies(pooled).join(pooled[kernel]).head()

,alpha_1.0,alpha_2.0,width_100,width_200,width_400,width_800,width_1600,eta_kappa_3e-06,eta_kappa_1e-05,eta_kappa_3e-05,eta_kappa_0.0001,eta_kappa_0.0003,eta_kappa_0.001,A_t,S_t,R_t
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.113876,1.318714,0.121258
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.106563,1.348164,0.120257
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.109269,1.354164,0.104921
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.112150,1.314184,0.115434
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.113564,1.413058,0.117339


## 1. The two formulas

**The linear model for when a run groks (Part 7).** This is a log-normal accelerated failure time (AFT) model, Equation 10 of the report. It is a linear regression on the log of the grokking step that also handles runs that never grok.

$$\log T_i = \beta_0 + \sum_j \beta_j \, \mathbb{1}[\text{run } i \text{ has setting value } j] + \beta_A A_i + \beta_S S_i + \beta_R R_i + \sigma \varepsilon_i, \qquad \varepsilon_i \sim N(0, 1)$$

- $T_i$ is the first step at which run $i$ reaches a test accuracy of 0.95.
- The sum runs over the values of $\alpha$, width and weight decay. Each value has its own indicator, so the model does not assume that width or $\alpha$ act along a straight line.
- $A_i$, $S_i$ and $R_i$ are the kernel numbers at step 2,000, scaled to mean 0 and standard deviation 1.
- A run that has not grokked by step 200,000 adds $P(T_i > 200{,}000)$ to the likelihood, as in Equation 11 of the report. Ordinary least squares would have to drop it, and that would drop the slowest runs.
- A coefficient reads as a multiplier on the grokking step. For example, $\beta_A = -0.3$ means that one standard deviation more of $A_t$ goes with a grokking step $e^{-0.3} \approx 0.74$ times as long.

**The logistic model for whether a run groks (Part 8).**

$$\operatorname{logit} P(\text{run } i \text{ groks by step } 200{,}000) = \gamma_0 + \sum_j \gamma_j \, \mathbb{1}[\text{run } i \text{ has setting value } j] + \gamma_A A_i + \gamma_S S_i + \gamma_R R_i$$

It has the same inputs. The AFT asks when a run groks, and the logistic model asks whether it groks within the budget.

Two details matter when reading the fits.

- scikit-learn's `LogisticRegression` adds a ridge penalty by default (`C = 1`). This does not affect scores such as the AUC. It shrinks the coefficients a little, so they should not be quoted as plain logistic coefficients.
- The AFT has a ridge penalty of 0.01. It is small enough to ignore when reading the coefficients, and it keeps them finite when a setting has no grokked run.

**Verdict: both formulas suit this data, and the report can write them as above.**

## 2. Lookahead

Lookahead means that an input carries information from after the moment we claim to predict from. Here that moment is step 2,000. The next cell checks four ways it could get in, and prints True for each check that passes.

In [3]:
pd.Series({
    'every input is read at step 2,000': (data['step'] == LANDMARK).all(),
    'no run reaches test accuracy 0.95 by step 2,000': events['T'].min() > LANDMARK,
    'no run has test accuracy above 0.2 at step 2,000': acc.loc[acc['step'] == LANDMARK, 'test_acc'].max() < 0.2,
    'every run has the same budget, so being censored says nothing extra': (runs['last_step'] == BUDGET).all(),
}, name='passes')

every input is read at step 2,000                                      True
no run reaches test accuracy 0.95 by step 2,000                        True
no run has test accuracy above 0.2 at step 2,000                       True
every run has the same budget, so being censored says nothing extra    True
Name: passes, dtype: bool

All four checks pass. Three more pass on reading the code.

- The kernel numbers are scaled with the mean and spread of the training runs only (`X.loc[train, scale]` in `hold_out_each_cell`). The held-out cell never sets the scale.
- No column measured at or after grokking goes into any model. Part 6 of `fitting.ipynb` lists those columns (`*_at_gen`, `*_final`, `A_peak`, `t_A*` and `weight_norm_growth`).
- The folds hold out whole cells. The other seeds of a run's cell are never in training while that run is scored.

**One possible leak to test.** The probe has 256 pairs, and 53 of them come from the test set. So $A_t$ uses the labels of 53 test pairs. That is information from the test set, even though it is read early. If the signal came from those 53 labels, a kernel built on the training pairs alone should predict worse. The next cell fits the settings + kernel AFT both ways on held-out cells and prints the two concordances. It also stores the predictions of the whole-probe model in `pred_both`, which Section 5c uses again.

In [4]:
train_pairs = ['A_t_train', 'S_c_train', 'R_c_train']
rows = {}
for label, df in sets.items():
    dummies = settings_dummies(df)
    df['pred_both'] = hold_out_each_cell(df, dummies.join(df[kernel]), scale=kernel)   # Section 5c uses this again
    df['pred_train_pairs'] = hold_out_each_cell(df, dummies.join(df[train_pairs]), scale=train_pairs)
    rows[label] = {'whole probe': concordance_index(df['T'], df['pred_both'], df['grokked']),
                   'training pairs only': concordance_index(df['T'], df['pred_train_pairs'], df['grokked'])}
pd.DataFrame(rows).round(3)

,weight decay 3e-5,no weight decay,all levels up to 1e-3
whole probe,0.940,0.968,0.901
training pairs only,0.946,0.964,0.901


**Verdict: there is no lookahead.** The kernel on the training pairs alone scores the same as the whole probe: 0.946 against 0.940 at weight decay 3e-5, 0.964 against 0.968 without weight decay, and 0.901 for both when pooled. So the 53 test labels are not doing the work.

One softer issue remains. Weight decay 3e-5 was picked as the main level after we saw that most of its runs grok. That choice was made by looking at the outcomes. The pooled set uses every level up to 1e-3, so the report should lead with the pooled set.

## 3. Multicollinearity

If two inputs move together, the model cannot tell which one deserves the credit. The fit still predicts well, but the single coefficients become unstable and can even change sign.

We look at two things.

- The first is the correlation between each pair of numbers. The rows marked "across runs" use every run. The rows marked "within cells" first subtract the mean of each cell, so they only see how the seeds of one cell differ. The within-cell Cox model of Part 7 uses only this second kind of variation.
- The second is the variance inflation factor (VIF). A VIF of 1 means no overlap, and a VIF above about 10 is the usual warning line.

The next cell prints the correlations for each set.

In [5]:
pairs = [('A_t', 'S_t'), ('A_t', 'R_t'), ('S_t', 'R_t')]
table = {}
for label, df in sets.items():
    within = df[kernel] - df.groupby('cell')[kernel].transform('mean')   # each run minus the mean of its cell
    table[label] = {**{('across runs', f'{a} and {b}'): df[a].corr(df[b]) for a, b in pairs},
                    **{('within cells', f'{a} and {b}'): within[a].corr(within[b]) for a, b in pairs}}
pd.DataFrame(table).round(2)

weight decay 3e-5  no weight decay  \
across runs  A_t and S_t               0.91             0.92   
             A_t and R_t               0.56             0.54   
             S_t and R_t               0.77             0.75   
within cells A_t and S_t               0.30             0.21   
             A_t and R_t               0.20             0.18   
             S_t and R_t              -0.08            -0.05   

                          all levels up to 1e-3  
across runs  A_t and S_t                   0.75  
             A_t and R_t                   0.36  
             S_t and R_t                   0.30  
within cells A_t and S_t                   0.09  
             A_t and R_t                   0.19  
             S_t and R_t                  -0.16

Across runs, `A_t` and `S_t` have a correlation of 0.91 at weight decay 3e-5 and 0.92 without weight decay. Within cells the same pair has a correlation of 0.30 or less, and no pair is above 0.30. The next cell prints the VIF of each kernel number, first with the kernel numbers alone and then with the settings indicators added.

In [6]:
table = {}
for label, df in sets.items():
    z = (df[kernel] - df[kernel].mean()) / df[kernel].std()
    for how, X in [('kernel only', z), ('with the settings', settings_dummies(df).join(z))]:
        X = sm.add_constant(X)
        table[(label, how)] = {k: variance_inflation_factor(X.to_numpy(), X.columns.get_loc(k)) for k in kernel}
pd.DataFrame(table).T.round(1)

A_t   S_t   R_t
weight decay 3e-5     kernel only         8.7  14.7   3.6
                      with the settings  30.5  42.0  11.8
no weight decay       kernel only         9.6  15.2   3.3
                      with the settings  28.0  36.9  12.2
all levels up to 1e-3 kernel only         2.4   2.3   1.2
                      with the settings   7.6  12.9   5.9

With the settings in the model, the VIFs of `A_t` and `S_t` reach 28 to 42 within one weight decay level, far above the warning line. Pooled, they are lower, from 6 to 13.

**Can the numbers be ranked by dropping one at a time?** An earlier version of Part 8 ranked the numbers by how much the AUC fell when each one was dropped, and it found that `R_t` mattered most. When two inputs overlap, dropping one of them costs little, because the other covers for it. So the next cell also drops `A_t` and `S_t` together. It prints the held-out AUC of the pooled logistic model for each choice.

In [7]:
dummies = settings_dummies(pooled)
drops = {'keep all three': kernel, 'drop R_t': ['A_t', 'S_t'], 'drop A_t': ['S_t', 'R_t'],
         'drop S_t': ['A_t', 'R_t'], 'drop A_t and S_t together': ['R_t'], 'drop all three': []}
auc = {}
for name, cols in drops.items():
    prob = hold_out_each_cell(pooled, dummies.join(pooled[cols]), scale=cols, model='logistic')
    auc[name] = roc_auc_score(pooled['grokked'], prob)
pd.Series(auc, name='held-out AUC, pooled').round(3)

keep all three               0.965
drop R_t                     0.958
drop A_t                     0.960
drop S_t                     0.964
drop A_t and S_t together    0.964
drop all three               0.950
Name: held-out AUC, pooled, dtype: float64

**Verdict: the numbers overlap a lot across runs and very little within cells.**

- Within one weight decay level, `A_t` and `S_t` are near twins across runs (r = 0.91 and 0.92). With the settings in the model their VIFs are 28 to 42. Pooled, the overlap is milder, with VIFs from 6 to 13.
- This explains the odd signs in Part 7, such as the positive coefficient of `S_t` in the pooled AFT. Section 5a shows that `S_t` even changes sign from one level to the next. So the AFT and logistic coefficients of `A_t`, `S_t` and `R_t` should not be read one at a time. The report should give the joint test of the three together.
- Within cells the overlap is small (r between -0.16 and 0.30). So the within-cell Cox coefficient of `A_t` in Part 7 is safe to read on its own, and it is the one to quote.
- Dropping one number at a time does not give a ranking. Each single drop moves the AUC by 0.007 or less. Dropping `A_t` and `S_t` together costs less than dropping `A_t` alone, which shows that these gaps are noise. The claim that `R_t` matters most has been removed from `fitting.ipynb`. What holds is that the three numbers together add about 0.015 to the AUC on top of the settings (0.950 to 0.965).

## 4. Are there enough grokked runs?

A common rule of thumb asks for at least 10 events per coefficient. For the AFT the events are the grokked runs. For the logistic model they are the runs of the smaller class.

The 5 seeds of a cell are near copies: Part 6 of `fitting.ipynb` found that 98 percent of the spread in the log grokking step lies between cells. So the honest count is closer to the number of cells than to the number of runs. The next cell counts both for each set.

In [8]:
rows = {}
for label, df in sets.items():
    k = settings_dummies(df).shape[1] + len(kernel)
    grok_cells = df.loc[df['grokked'] == 1, 'cell'].nunique()
    rows[label] = {'coefficients': k, 'cells': df['cell'].nunique(),
                   'grokked runs': df['grokked'].sum(), 'runs not grokked': (1 - df['grokked']).sum(),
                   'cells with a grokked run': grok_cells,
                   'cells with a run not grokked': df.loc[df['grokked'] == 0, 'cell'].nunique(),
                   'grokked runs per coefficient': df['grokked'].sum() / k,
                   'cells with a grokked run per coefficient': grok_cells / k}
pd.DataFrame(rows).round(1)

,weight decay 3e-5,no weight decay,all levels up to 1e-3
coefficients,10.0,10.0,16.0
cells,18.0,18.0,126.0
grokked runs,86.0,23.0,355.0
runs not grokked,4.0,67.0,275.0
cells with a grokked run,18.0,5.0,75.0
cells with a run not grokked,1.0,14.0,57.0
grokked runs per coefficient,8.6,2.3,22.2
cells with a grokked run per coefficient,1.8,0.5,4.7


**Verdict: the pooled set is large enough, and the single levels are thin.**

- **Pooled.** There are 355 grokked runs for 16 coefficients, about 22 per coefficient. Counting cells instead, 75 cells have a grokked run, about 5 per coefficient. That is enough for the joint tests and the held-out scores.
- **Weight decay 3e-5.** There are 86 grokked runs, but only 18 cells for 10 coefficients. This set works for when a run groks, and the held-out concordance guards against overfitting. It cannot answer whether a run groks, because the 4 runs that do not grok all sit in one cell. That is why Part 8 skips this set.
- **No weight decay.** There are 23 grokked runs in 5 cells for 10 coefficients. That is too few to say anything about single coefficients, so this set should only be a side check. Its high concordance (0.97) comes from the settings alone. Part 9 of `fitting.ipynb` looks at what more runs at this level would need to look like.

## 5. Weight decay: is it a constant effect, and is it fair to swap levels?

This section asks three questions.

- **5a.** Does each kernel number have the same effect at every weight decay level?
- **5b.** Does weight decay only move the grokking time, or does it also change the spread?
- **5c.** If we train on some levels, can we predict a level the model has never seen?

### 5a. Is the kernel effect the same at every level?

The next cell adds an interaction for each kernel number with each weight decay level to the pooled AFT, and tests whether those 18 extra coefficients improve the fit. The test treats the seeds as independent runs, which makes it too confident. So it also prints the p-value after cutting the evidence by a factor of 5, which is roughly what we would get if each cell counted as one run.

In [9]:
z = (pooled[kernel] - pooled[kernel].mean()) / pooled[kernel].std()
dummies = settings_dummies(pooled)
wd_cols = [c for c in dummies if c.startswith('eta_kappa_')]
X_base = dummies.join(z)

# one column per kernel number and weight decay level, such as 'A_t x eta_kappa_3e-05'
interactions = pd.DataFrame({f'{k} x {w}': z[k] * dummies[w] for k in kernel for w in wd_cols})
print(list(interactions.columns[:3]), '...')

base = LogNormalAFTFitter(penalizer=0.01).fit(X_base.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
with_inter = LogNormalAFTFitter(penalizer=0.01).fit(X_base.join(interactions).assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
lr = 2 * (with_inter.log_likelihood_ - base.log_likelihood_)
pd.Series({'extra coefficients': interactions.shape[1], 'likelihood ratio': round(lr, 1),
           'p': f'{chi2.sf(lr, interactions.shape[1]):.1g}',
           'p with the evidence cut by 5': f'{chi2.sf(lr / 5, interactions.shape[1]):.1g}'})

['A_t x eta_kappa_3e-06', 'A_t x eta_kappa_1e-05', 'A_t x eta_kappa_3e-05'] ...


extra coefficients                 18
likelihood ratio                279.0
p                               1e-48
p with the evidence cut by 5    1e-05
dtype: object

The 18 interactions improve the fit by a likelihood ratio of 279. Even with the evidence cut by 5, the p-value is 1e-05. So the kernel numbers do not have the same effect at every level.

The next cell shows what differs. It fits the AFT on each level by itself, with indicators for `alpha` and width, and prints the three kernel coefficients. A star marks p < 0.05.

In [10]:
per_level = {}
for level, df in pooled.groupby('eta_kappa'):
    z_level = (df[kernel] - df[kernel].mean()) / df[kernel].std()
    fit = LogNormalAFTFitter(penalizer=0.01).fit(settings_dummies(df).join(z_level).assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
    s = fit.summary.loc['mu_'].loc[kernel]
    per_level[(f'{level:g}', df['grokked'].sum())] = s['coef'].round(2).astype(str) + np.where(s['p'] < 0.05, ' *', '')
per_level = pd.DataFrame(per_level).T
per_level.index.names = ['weight decay', 'grokked runs']
per_level

,covariate,A_t,S_t,R_t
weight decay,grokked runs,,,
0,23,-0.3,-0.87,-0.06
3e-06,40,-0.28,-0.18,0.41 *
1e-05,64,-0.35 *,-0.43 *,0.26 *
3e-05,86,-0.33 *,0.13,-0.28 *
0.0001,77,-0.03,0.82 *,-0.56 *
0.0003,50,0.1,2.03 *,-0.41 *
0.001,15,-0.23,-1.17 *,0.24


The coefficient of `A_t` stays near -0.3 from no weight decay up to 3e-5 and then fades toward 0. `S_t` and `R_t` change sign between levels. So the pooled AFT coefficients are an average over levels that disagree.

The within-cell Cox model is the main result of Part 7, so we ask the same question of it. The next cell adds one interaction of `A_t` with each weight decay level to the within-cell Cox model. It tests whether the effect of `A_t` among the seeds of one cell changes with the level.

In [11]:
cox_table = pooled[['T', 'grokked', 'cell']].join(z)
cox_inter = cox_table.join(pd.DataFrame({f'A_t x {w}': z['A_t'] * dummies[w] for w in wd_cols}))
cox_base = CoxPHFitter().fit(cox_table, 'T', 'grokked', strata='cell')
cox_full = CoxPHFitter().fit(cox_inter, 'T', 'grokked', strata='cell')
lr = 2 * (cox_full.log_likelihood_ - cox_base.log_likelihood_)
pd.Series({'extra coefficients': len(wd_cols), 'likelihood ratio': round(lr, 1), 'p': round(chi2.sf(lr, len(wd_cols)), 3)})

extra coefficients    6.000
likelihood ratio      1.800
p                     0.935
dtype: float64

The 6 interactions improve the fit by a likelihood ratio of only 1.8 (p = 0.935). So among the seeds of one cell, the effect of `A_t` is the same at every weight decay level.

### 5b. Does weight decay only move the time?

The AFT above gives each weight decay level its own shift in the log grokking step, but the same spread $\sigma$ for all of them. The next cell lets $\sigma$ differ by level with the `ancillary` argument of lifelines. It tests whether that improves the fit and prints $\sigma$ at each level.

In [12]:
# `ancillary` lets the spread sigma have its own value at each weight decay level
spread = LogNormalAFTFitter(penalizer=0.01).fit(X_base.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked',
                                                ancillary=dummies[wd_cols])
lr = 2 * (spread.log_likelihood_ - base.log_likelihood_)
sig = spread.summary.loc['sigma_', 'coef']
sigma_by_level = pd.Series({'0': np.exp(sig['Intercept']),
                            **{c.split('_')[-1]: np.exp(sig['Intercept'] + sig[c]) for c in wd_cols}})
print(f'likelihood ratio {lr:.1f} on {len(wd_cols)} extra coefficients, p = {chi2.sf(lr, len(wd_cols)):.1g}')
sigma_by_level.rename('sigma, spread of log T').round(2)

likelihood ratio 169.2 on 6 extra coefficients, p = 7e-34


0         1.66
3e-06     1.11
1e-05     0.50
3e-05     0.19
0.0001    0.50
0.0003    1.27
0.001     1.95
Name: sigma, spread of log T, dtype: float64

Letting the spread differ improves the fit by a likelihood ratio of 169 on 6 coefficients (p = 7e-34). The spread is smallest at 3e-5, where $\sigma$ is 0.19, and it grows on both sides, to 1.66 without weight decay and 1.95 at 1e-3.

### 5c. Can the model predict a weight decay level it has never seen?

The next cell holds out one whole weight decay level, trains on the other six, and predicts the held-out level. A model without weight decay indicators has to work out the effect of the level from what it sees. Each model is scored in two ways.

- **Across all runs.** This asks whether the model puts the held-out levels in the right place relative to each other.
- **Within a level, averaged.** This asks whether the model ranks the runs inside a level correctly.

The last two rows are the leave-one-cell-out scores, where every level is in training. They are there for comparison.

In [13]:
alpha_width = pd.get_dummies(pooled[['alpha', 'width']].astype('category'), drop_first=True, dtype=float)
rows = {}
for name, use_settings, use_kernel in [('alpha + width', True, False), ('kernel only', False, True),
                                       ('alpha + width + kernel', True, True)]:
    pred = pd.Series(index=pooled.index, dtype=float)
    for level in pooled['eta_kappa'].unique():
        train = pooled['eta_kappa'] != level
        test = pooled['eta_kappa'] == level
        X = pd.DataFrame(index=pooled.index)
        if use_settings:
            X = X.join(alpha_width)
        if use_kernel:
            X = X.join((pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std())
        fit = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
        pred[test] = fit.predict_median(X[test]).to_numpy().ravel()
    pooled[f'pred_{name}'] = pred
    rows[f'{name}, level held out'] = pred

rows['all settings, cell held out'] = hold_out_each_cell(pooled, settings_dummies(pooled), scale=[])
rows['all settings + kernel, cell held out'] = pooled['pred_both']   # from Section 2

scores = {}
for name, pred in rows.items():
    within = [concordance_index(g['T'], pred[g.index], g['grokked']) for _, g in pooled.groupby('eta_kappa')]
    scores[name] = {'across all runs': concordance_index(pooled['T'], pred, pooled['grokked']),
                    'within a level, averaged': np.mean(within)}
pd.DataFrame(scores).T.round(3)

,across all runs,"within a level, averaged"
"alpha + width, level held out",0.765,0.923
"kernel only, level held out",0.766,0.900
"alpha + width + kernel, level held out",0.729,0.919
"all settings, cell held out",0.877,0.918
"all settings + kernel, cell held out",0.901,0.924


**Verdict: weight decay is not a constant effect. Each level should keep its own term, and the levels should not be swapped.**

- **5a, across cells.** The kernel numbers do not have the same effect at every level (p = 1e-5 even with the evidence cut by 5). `A_t` sits at about -0.3 from no weight decay up to 3e-5, then fades to near 0 from 1e-4 upward. `S_t` and `R_t` change sign between levels. So the pooled AFT coefficients are an average over levels that disagree.
- **5a, within cells.** Here the answer is the opposite. The effect of `A_t` among the seeds of one cell is the same at every level (p = 0.94). This is the clearest finding of the audit: a seed with higher alignment at step 2,000 groks sooner than the other seeds of its cell, whatever the weight decay.
- **5b.** Weight decay changes the spread as well as the typical time (p = 7e-34). The spread of log T is smallest at 3e-5 (0.19) and grows on both sides, to 1.66 without weight decay and 1.95 at 1e-3. That is why the Kaplan-Meier curves in Part 5 cross.
- **5c.** A model that has not seen a level can still rank the runs inside it (0.90 to 0.92 within a level). It cannot place that level against the others: across all runs the concordance drops from 0.88 and 0.90 to between 0.73 and 0.77. The kernel at step 2,000 does not carry the effect of weight decay either. Adding it to `alpha` and width makes the cross-level score worse (0.729).

**Is swapping levels fair?** Training on 3e-5 and testing on no weight decay mixes two things: whether the kernel predicts grokking, and how much the level itself moves the answer. The fair designs are the ones already in `fitting.ipynb`. One is to fit one level at a time. The other is to pool every level with one indicator per level and hold out cells. In the paper, say that weight decay sets its own baseline time and spread, and that the model is not meant to predict a level it has not seen.

Part 9 of `fitting.ipynb` runs the same test for `alpha`, and there the kernel does carry the effect. With `alpha` held out, the kernel alone reaches a concordance of 0.80, against 0.55 for width and weight decay.

## 6. Is a large L1 model worth it?

The idea is to put every number logged at step 2,000 into a logistic model with an L1 penalty, and let the penalty pick the ones that matter. There are about 100 of them. Anything measured on the test set is left out (`test_*`, `*_test`, `krr_*` and `A_full`), so no input uses test labels.

The penalty strength is picked inside each training fold with 5 inner folds. All models in the next table use the same 10 outer folds, and each fold holds out whole cells. The cell prints the held-out AUC of each model on the pooled set.

In [14]:
skip = {'run_id', 'step', 'log10_step1', 'dt_prev', 'on_linear_grid', 'on_log_grid', 'wall_seconds',
        'stationary', 'kernel_degenerate', 'A_full'}
l1_features = [c for c in at_landmark.columns
               if c not in skip and not c.startswith(('test_', 'krr_')) and not c.endswith('_test')
               and pooled[c].notna().all() and pooled[c].std() > 0]
print(len(l1_features), 'features go into the L1 model')

Cs = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

def l1(C):
    return LogisticRegression(l1_ratio=1, C=C, solver='liblinear', max_iter=5000, random_state=0)

def pick_C(X, y, groups):
    """Pick the penalty strength by 5 inner folds of whole cells."""
    inner = list(GroupKFold(5).split(X, y, groups))
    return max(Cs, key=lambda C: np.mean([roc_auc_score(y.iloc[b], l1(C).fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1])
                                          for a, b in inner]))

folds = list(GroupKFold(10).split(pooled, groups=pooled['cell']))
dummies = settings_dummies(pooled)
candidates = {'settings only': (dummies, []),
              'the 3 kernel numbers only': (pooled[kernel], kernel),
              'settings + the 3 kernel numbers': (dummies.join(pooled[kernel]), kernel),
              f'L1 on all {len(l1_features)} numbers': (pooled[l1_features], l1_features)}
auc = {}
for name, (X, scale) in candidates.items():
    prob = np.zeros(len(pooled))
    for train, test in folds:
        X_fold = X.copy()
        X_fold[scale] = (X[scale] - X.iloc[train][scale].mean()) / X.iloc[train][scale].std()
        y_train = pooled['grokked'].iloc[train]
        if name.startswith('L1'):
            model = l1(pick_C(X_fold.iloc[train], y_train, pooled['cell'].iloc[train]))
        else:
            model = LogisticRegression(max_iter=1000)
        prob[test] = model.fit(X_fold.iloc[train], y_train).predict_proba(X_fold.iloc[test])[:, 1]
    auc[name] = roc_auc_score(pooled['grokked'], prob)
pd.Series(auc, name='held-out AUC, pooled').round(3)

108 features go into the L1 model


settings only                      0.948
the 3 kernel numbers only          0.886
settings + the 3 kernel numbers    0.960
L1 on all 108 numbers              0.983
Name: held-out AUC, pooled, dtype: float64

The L1 model reaches a held-out AUC of 0.983, against 0.960 for the settings and the three kernel numbers on the same folds. The settings + kernel score here is a little lower than the 0.965 of Section 3, because these folds hold out 10 groups of cells instead of one cell at a time.

To see which numbers the L1 model picks, and whether it picks the same ones every time, the next cell redraws the cells with replacement 100 times, refits, and counts how often each number survives the penalty. The last column is the share of each number's spread that the weight decay level alone explains.

The table also shows some near copies of the report numbers: `A_t_train` and `A_first_logit` for `A_t` (r = 0.99), `S_full` and `S_W1` for `S_t` (r = 1.00), and `share_k0` and `R_full` for `R_t` (r = 1.00).

In [15]:
X_all = (pooled[l1_features] - pooled[l1_features].mean()) / pooled[l1_features].std()
y_all = pooled['grokked']
best_C = pick_C(X_all, y_all, pooled['cell'])

rng = np.random.default_rng(0)
cell_rows = pooled.groupby('cell').indices        # cell name -> row positions
picked = pd.Series(0.0, index=X_all.columns)
for _ in range(100):
    redraw = rng.choice(list(cell_rows), len(cell_rows))     # cells drawn with replacement
    rows = np.concatenate([cell_rows[c] for c in redraw])
    picked += l1(best_C).fit(X_all.iloc[rows], y_all.iloc[rows]).coef_.ravel() != 0
picked /= 100

l1_table = pd.DataFrame({'picked in this share of redraws': picked,
                         'share of spread set by weight decay': X_all.groupby(pooled['eta_kappa']).transform('mean').var() / X_all.var()})
l1_table = l1_table.sort_values('picked in this share of redraws', ascending=False)
l1_table['rank'] = np.arange(1, len(l1_table) + 1)
print(f'penalty C = {best_C}; numbers picked in at least half the redraws: {(picked >= 0.5).sum()}')
copies = ['A_t_train', 'A_first_logit', 'S_full', 'S_W1', 'share_k0', 'R_full']
pd.concat({'top ten': l1_table.head(10), 'the report numbers': l1_table.loc[kernel],
           'near copies of them': l1_table.loc[copies]}).round(2)

penalty C = 3; numbers picked in at least half the redraws: 25


picked in this share of redraws  \
top ten             train_acc                                        1.00   
                    decay_residual                                   1.00   
                    eig_gap_top88                                    0.97   
                    grad_weight_cos                                  0.97   
                    grad_norm_W1                                     0.97   
                    fourier_dW_c_dcshare                             0.90   
                    fourier_dW_a_dcshare                             0.87   
                    fourier_dW_match                                 0.87   
                    dA_step                                          0.86   
                    D                                                0.81   
the report numbers  A_t                                              0.00   
                    S_t                                              0.04   
                    R_t                                              0.03   
near copies of them A_t_train                                        0.10   
                    A_first_logit                                    0.07   
                    S_full                                           0.04   
                    S_W1                                             0.02   
                    share_k0                                         0.02   
                    R_full                                           0.04   

                                          share of spread set by weight decay  \
top ten             train_acc                                            0.74   
                    decay_residual                                       0.92   
                    eig_gap_top88                                        0.00   
                    grad_weight_cos                                      0.59   
                    grad_norm_W1                                         0.18   
                    fourier_dW_c_dcshare                                 0.76   
                    fourier_dW_a_dcshare                                 0.68   
                    fourier_dW_match                                     0.13   
                    dA_step                                              0.01   
                    D                                                    0.01   
the report numbers  A_t                                                  0.18   
                    S_t                                                  0.56   
                    R_t                                                  0.07   
near copies of them A_t_train                                            0.16   
                    A_first_logit                                        0.20   
                    S_full                                               0.56   
                    S_W1                                                 0.49   
                    share_k0                                             0.07   
                    R_full                                               0.07   

                                          rank  
top ten             train_acc                1  
                    decay_residual           2  
                    eig_gap_top88            3  
                    grad_weight_cos          4  
                    grad_norm_W1             5  
                    fourier_dW_c_dcshare     6  
                    fourier_dW_a_dcshare     7  
                    fourier_dW_match         8  
                    dA_step                  9  
                    D                       10  
the report numbers  A_t                    101  
                    S_t                     75  
                    R_t                     80  
near copies of them A_t_train               61  
                    A_first_logit           71  
                    S_full                  76  
                    S_W1                    87 

**Verdict: it is not worth a section. It deserves one sentence in the appendix at most.**

- It does score higher: 0.983 against 0.960 for the settings and the three kernel numbers, on the same folds.
- Its picks show why. Three of the top four are the training accuracy at step 2,000, `decay_residual` and `grad_weight_cos`. Weight decay alone sets 59 to 92 percent of their spread. So the L1 model mostly rebuilds the settings from inside the network, and adds how far memorisation has got.
- It drops `A_t`, `S_t` and `R_t` in almost every redraw, and their near copies too, which it picks in 10 percent of redraws or fewer. Once weight decay and the training accuracy are in, the kernel numbers have little left to add about whether a run groks.
- Only about 25 numbers survive in at least half of the redraws. Below the top five, the picks change from one redraw to the next.

The report asks about `A_t`, `S_t` and `R_t`. The L1 model answers a different question, which is what the best early alarm for grokking is. A proper version of that would need its own section, its own survival model and its own checks, so it is out of scope. A sentence for the appendix could read: "An L1 logistic model over all 108 numbers logged at step 2,000 reaches a held-out AUC of 0.98. Its top picks track weight decay and training accuracy, and it rarely selects A_t, S_t or R_t."

## 7. One kernel number at a time

This is the same check as the last table of Part 8 of `fitting.ipynb`, with ranges added. Each logistic model gets one of the three numbers, and we score it on held-out cells. We do it two ways.

- **Alone.** Only that number goes in. This shows everything the number knows about whether a run groks, including what it knows about the settings.
- **On top of the settings.** The number goes in with the `alpha`, width and weight decay indicators. This shows what it adds beyond the settings.

Fitting one number at a time avoids the overlap problem of Section 3, because each model holds only one kernel number. The 3e-5 set is skipped for the same reason as in Part 8: its 4 runs that do not grok all sit in one cell. The next cell prints the AUC, the balanced accuracy and the plain accuracy of each model.

In [16]:
rows, single_probs = {}, {}
for label in ['no weight decay', POOLED]:
    df = sets[label]
    dummies = settings_dummies(df)
    for how in ['alone', 'on top of the settings']:
        models = {'A_t only': ['A_t'], 'S_t only': ['S_t'], 'R_t only': ['R_t'], 'all three': kernel}
        if how == 'on top of the settings':
            models = {'settings only': [], **models}
        for name, cols in models.items():
            X = df[cols] if how == 'alone' else dummies.join(df[cols])
            p = hold_out_each_cell(df, X, scale=cols, model='logistic')
            single_probs[(label, how, name)] = p
            rows[(label, how, name)] = {'AUC': roc_auc_score(df['grokked'], p),
                                        'balanced accuracy': balanced_accuracy_score(df['grokked'], p > 0.5),
                                        'accuracy': accuracy_score(df['grokked'], p > 0.5)}
pd.DataFrame(rows).T.round(3)

AUC  \
no weight decay       alone                  A_t only       0.695   
                                             S_t only       0.879   
                                             R_t only       0.975   
                                             all three      0.969   
                      on top of the settings settings only  0.855   
                                             A_t only       0.864   
                                             S_t only       0.952   
                                             R_t only       0.966   
                                             all three      0.992   
all levels up to 1e-3 alone                  A_t only       0.868   
                                             S_t only       0.763   
                                             R_t only       0.551   
                                             all three      0.880   
                      on top of the settings settings only  0.950   
                                             A_t only       0.959   
                                             S_t only       0.949   
                                             R_t only       0.964   
                                             all three      0.965   

                                                            balanced accuracy  \
no weight decay       alone                  A_t only                   0.709   
                                             S_t only                   0.730   
                                             R_t only                   0.826   
                                             all three                  0.876   
                      on top of the settings settings only              0.680   
                                             A_t only                   0.724   
                                             S_t only                   0.768   
                                             R_t only                   0.826   
                                             all three                  0.826   
all levels up to 1e-3 alone                  A_t only                   0.807   
                                             S_t only                   0.656   
                                             R_t only                   0.544   
                                             all three                  0.808   
                      on top of the settings settings only              0.864   
                                             A_t only                   0.871   
                                             S_t only                   0.867   
                                             R_t only                   0.882   
                                             all three                  0.888   

                                                            accuracy  
no weight decay       alone                  A_t only          0.800  
                                             S_t only          0.811  
                                             R_t only          0.911  
                                             all three         0.922  
                      on top of the settings settings only     0.800  
                                             A_t only          0.844  
                                             S_t only          0.867  
                                             R_t only          0.911  
                                             all three         0.911  
all levels up to 1e-3 alone                  A_t only          0.808  
                                             S_t only          0.683  
                                             R_t only          0.551  
                                             all three         0.805  
                      on top of the settings settings only     0.867  
                                             A_t only          0.873  
                                             S_t only          0.867  
   

**Is a gap between two numbers real?** The table above gives one AUC per model, and some of the gaps are small. The next cell redraws the cells with replacement 1,000 times and recomputes the AUC gap between two models each time. It prints the gap and the middle 95 percent of the redrawn gaps. If that range does not include 0, the winner is clear.

In [17]:
rows = {}
for label in ['no weight decay', POOLED]:
    df = sets[label]
    y = df['grokked'].to_numpy()
    cell_rows = list(df.groupby('cell').indices.values())
    for how in ['alone', 'on top of the settings']:
        for a, b in [('A_t only', 'S_t only'), ('A_t only', 'R_t only'), ('S_t only', 'R_t only')]:
            pa = single_probs[(label, how, a)].to_numpy()
            pb = single_probs[(label, how, b)].to_numpy()
            rng, gaps = np.random.default_rng(0), []
            while len(gaps) < 1000:
                redraw = np.concatenate([cell_rows[i] for i in rng.integers(len(cell_rows), size=len(cell_rows))])
                if y[redraw].min() < y[redraw].max():       # skip a redraw that holds one class only
                    gaps.append(roc_auc_score(y[redraw], pa[redraw]) - roc_auc_score(y[redraw], pb[redraw]))
            low, high = np.percentile(gaps, [2.5, 97.5])
            rows[(label, how, f'{a} minus {b}')] = {'AUC gap': roc_auc_score(y, pa) - roc_auc_score(y, pb),
                                                    'low': low, 'high': high}
pd.DataFrame(rows).T.round(3)

AUC gap  \
no weight decay       alone                  A_t only minus S_t only   -0.184   
                                             A_t only minus R_t only   -0.280   
                                             S_t only minus R_t only   -0.096   
                      on top of the settings A_t only minus S_t only   -0.088   
                                             A_t only minus R_t only   -0.101   
                                             S_t only minus R_t only   -0.014   
all levels up to 1e-3 alone                  A_t only minus S_t only    0.105   
                                             A_t only minus R_t only    0.317   
                                             S_t only minus R_t only    0.212   
                      on top of the settings A_t only minus S_t only    0.010   
                                             A_t only minus R_t only   -0.005   
                                             S_t only minus R_t only   -0.014   

                                                                        low  \
no weight decay       alone                  A_t only minus S_t only -0.419   
                                             A_t only minus R_t only -0.621   
                                             S_t only minus R_t only -0.300   
                      on top of the settings A_t only minus S_t only -0.279   
                                             A_t only minus R_t only -0.343   
                                             S_t only minus R_t only -0.131   
all levels up to 1e-3 alone                  A_t only minus S_t only  0.050   
                                             A_t only minus R_t only  0.208   
                                             S_t only minus R_t only  0.101   
                      on top of the settings A_t only minus S_t only -0.004   
                                             A_t only minus R_t only -0.019   
                                             S_t only minus R_t only -0.028   

                                                                       high  
no weight decay       alone                  A_t only minus S_t only -0.010  
                                             A_t only minus R_t only  0.009  
                                             S_t only minus R_t only  0.040  
                      on top of the settings A_t only minus S_t only  0.010  
                                             A_t only minus R_t only  0.072  
                                             S_t only minus R_t only  0.111  
all levels up to 1e-3 alone                  A_t only minus S_t only  0.169  
                                             A_t only minus R_t only  0.428  
                                             S_t only minus R_t only  0.323  
                      on top of the settings A_t only minus S_t only  0.026  
                                             A_t only minus R_t only  0.010  
                                             S_t only minus R_t only -0.003

**Verdict: this check is worth keeping. It is cheap, it avoids the overlap problem, and its answer depends on which question is asked.**

- **Alone, pooled.** `A_t` is clearly the most useful number (AUC 0.868), then `S_t` (0.763). `R_t` is close to a coin toss (0.551). Every gap is clear, since none of the ranges includes 0.
- **Alone, no weight decay.** The order is reversed: `R_t` 0.975, `S_t` 0.879, `A_t` 0.695. This set has only 5 cells with a grokked run, and two of the three ranges include 0. No conclusion should rest on it.
- **On top of the settings, pooled.** No number stands out. The settings alone reach 0.950. `R_t` adds 0.014, `A_t` adds 0.009 and `S_t` adds nothing. The gaps between the numbers are 0.015 or less, and only one range (`S_t` against `R_t`) just misses 0.
- `A_t` does well alone but adds little on top because a good part of what it knows at step 2,000 is which cell the run is in. Once the settings are in, that part is used up.
- The comparison should use the AUC or the balanced accuracy. Plain accuracy depends on how many runs grok in the set: always guessing "does not grok" already gets 0.744 without weight decay.

For the paper, this means the following. On its own, `A_t` is the most informative of the three about whether a run groks. Beyond the settings, the three add a similar small amount, and the data cannot rank them. For when a run groks, the within-cell Cox model of Part 7 gives the clearest ranking, and there `A_t` carries the signal.

## 8. Interaction terms

An interaction asks whether the effect of one number depends on another. For example, `A_t x S_t` asks whether high alignment helps more or less when the kernel has also grown. We add the three pairwise products (`A_t x S_t`, `A_t x R_t` and `S_t x R_t`) to the settings + kernel model and ask three questions.

- Do they improve the held-out scores?
- Do they pass a likelihood ratio test in the pooled AFT, and do they still pass once we also add squares, or the kernel by weight decay terms of Section 5a? Two near-twin inputs make a product that looks a lot like a square, so a product can be picking up a curve in one number. Section 5a also showed that the kernel effects differ by level, and a product could be standing in for that.
- Do they show up within cells, where the settings drop out?

### 8a. Held-out scores

The next cell reruns the hold-out loop of Part 7 with the products added, and then with the squares added as well. It prints the held-out concordance of the AFT and the held-out AUC of the logistic model. The 3e-5 set gets no AUC, for the reason given in Section 7.

In [18]:
pairs = [('A_t', 'S_t'), ('A_t', 'R_t'), ('S_t', 'R_t')]
rows = {}
for label, df in sets.items():
    dummies = settings_dummies(df)
    classify = label != 'weight decay 3e-5'      # its 4 failures sit in one cell
    for extra in ['main effects only', '+ products', '+ products + squares']:
        pred = pd.Series(index=df.index, dtype=float)
        prob = pd.Series(index=df.index, dtype=float)
        for cell in df['cell'].unique():
            train = df['cell'] != cell
            test = df['cell'] == cell
            z_fold = (df[kernel] - df.loc[train, kernel].mean()) / df.loc[train, kernel].std()
            X = dummies.join(z_fold)
            if extra != 'main effects only':
                for a, b in pairs:
                    X[f'{a} x {b}'] = z_fold[a] * z_fold[b]
            if extra == '+ products + squares':
                for k in kernel:
                    X[f'{k}^2'] = z_fold[k] ** 2
            aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
            pred[test] = aft.predict_median(X[test]).to_numpy().ravel()
            if classify:
                prob[test] = LogisticRegression(max_iter=2000).fit(X[train], df.loc[train, 'grokked']).predict_proba(X[test])[:, 1]
        rows[(label, extra)] = {'concordance (AFT)': concordance_index(df['T'], pred, df['grokked']),
                                'AUC (logistic)': roc_auc_score(df['grokked'], prob) if classify else np.nan}
pd.DataFrame(rows).T.round(3)

concordance (AFT)  AUC (logistic)
weight decay 3e-5     main effects only                 0.940             NaN
                      + products                        0.944             NaN
                      + products + squares              0.948             NaN
no weight decay       main effects only                 0.968           0.992
                      + products                        0.970           0.981
                      + products + squares              0.935           0.933
all levels up to 1e-3 main effects only                 0.901           0.965
                      + products                        0.909           0.972
                      + products + squares              0.911           0.973

Pooled, the products raise the concordance from 0.901 to 0.909 and the AUC from 0.965 to 0.972. Within a single level they help little, and without weight decay the squares make both scores worse.

### 8b. Do the products survive in the pooled AFT?

Each row of the next table adds the three products on top of something else and tests them with a likelihood ratio test. The first row adds them on top of nothing else, the second on top of the squares, and the third on top of the kernel by weight decay terms of Section 5a. A star marks p < 0.05 for a single product.

In [19]:
products = pd.DataFrame({f'{a} x {b}': z[a] * z[b] for a, b in pairs})
squares = pd.DataFrame({f'{k}^2': z[k] ** 2 for k in kernel})
backgrounds = {'nothing else': X_base.iloc[:, :0], 'the squares': squares,
               'the kernel x weight decay terms': interactions}
rows = {}
for name, background in backgrounds.items():
    X_without = X_base.join(background)
    X_with = X_without.join(products)
    without = LogNormalAFTFitter(penalizer=0.01).fit(X_without.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
    with_products = LogNormalAFTFitter(penalizer=0.01).fit(X_with.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
    lr = 2 * (with_products.log_likelihood_ - without.log_likelihood_)
    c = with_products.summary.loc['mu_'].loc[products.columns]
    rows[f'on top of {name}'] = {**{p: f"{c.loc[p, 'coef']:.2f}{' *' if c.loc[p, 'p'] < 0.05 else ''}" for p in products},
                                 'p, all three products': f'{chi2.sf(lr, 3):.1g}',
                                 'p with the evidence cut by 5': f'{chi2.sf(lr / 5, 3):.1g}'}
pd.DataFrame(rows).T

,A_t x S_t,A_t x R_t,S_t x R_t,"p, all three products",p with the evidence cut by 5
on top of nothing else,0.74 *,0.23 *,-0.37 *,1e-23,7e-05
on top of the squares,0.57 *,-0.02,-0.64 *,2e-08,0.05
on top of the kernel x weight decay terms,0.62 *,0.07,-0.12,1e-09,0.03


On their own the products pass easily (p = 1e-23). With the squares or the kernel by weight decay terms in the model, and the evidence cut by 5, p rises to between 0.03 and 0.05. Only `A_t x S_t` keeps a star in every row.

### 8c. Within cells

The last test adds the three products to the within-cell Cox model, where the settings drop out. The next cell prints the likelihood ratio test and the coefficients.

In [20]:
within_base = CoxPHFitter().fit(cox_table, 'T', 'grokked', strata='cell')
within_products = CoxPHFitter().fit(cox_table.join(products), 'T', 'grokked', strata='cell', robust=True)
lr = 2 * (within_products.log_likelihood_ - within_base.log_likelihood_)
print(f'likelihood ratio {lr:.1f} on 3 extra coefficients, p = {chi2.sf(lr, 3):.2f}')
within_products.summary[['coef', 'p']].round(3)

likelihood ratio 0.9 on 3 extra coefficients, p = 0.83


,coef,p
covariate,,
A_t,1.614,0.005
S_t,2.353,0.532
R_t,-0.001,1.000
A_t x S_t,0.428,0.407
A_t x R_t,-0.250,0.442
S_t x R_t,-0.721,0.535


The products improve the within-cell fit by a likelihood ratio of only 0.9 (p = 0.83), and none of them has a p-value below 0.4. `A_t` keeps its coefficient of about 1.6 (p = 0.005).

**Verdict: the products add a little when pooled, nothing within a level and nothing within cells. They should stay out of the headline.**

- **Held-out.** Pooled, the products raise the concordance from 0.901 to 0.909 and the AUC from 0.965 to 0.972. Within a single level they do not help. Without weight decay, adding the squares makes things worse (0.968 to 0.935), which is overfitting on 5 grokked cells.
- **Pooled AFT.** The products pass easily on their own (p = 1e-23). Once the squares or the kernel by weight decay terms are in, and the evidence is cut by 5 for the near-copy seeds, p is only 0.03 to 0.05. `A_t x R_t` disappears once the squares are in. `A_t x S_t` is the only product that stays in every row.
- **Within cells.** Nothing passes (p = 0.83). Among the seeds of one cell, the effect of `A_t` does not depend on `S_t` or `R_t`.

So the products mostly stand in for curves in single numbers and for the weight decay differences of Section 5a. The interaction with real support is kernel by weight decay (Section 5a). `A_t x S_t` could get one line in the appendix as a borderline result.

## Summary

### What we did

We reran the models of Parts 5 to 8 of `fitting.ipynb` on the same data and put them through eight checks. Every score holds out whole cells, and every input is the kernel at step 2,000. The table gives the verdict of each check in one line. The sections below it give what each check did, what it found, and where it belongs in the report.

| Check | Verdict |
| --- | --- |
| 1. Formulas | Both models suit the data. |
| 2. Lookahead | There is none. The kernel on the training pairs scores the same as the whole probe. |
| 3. Multicollinearity | `A_t` and `S_t` overlap across runs, so their single coefficients should not be read. Within cells they do not overlap. |
| 4. Sample size | The pooled set is large enough. The level without weight decay is too thin for anything but a side check. |
| 5. Weight decay | It is not a constant effect. Each level needs its own term, and levels should not be swapped. The within-cell effect of `A_t` is the same at every level. |
| 6. Large L1 model | It scores higher by rebuilding weight decay, and it is out of scope. |
| 7. One number at a time | Alone, `A_t` is the most useful. On top of the settings, the three cannot be ranked. |
| 8. Interactions | They add a little when pooled and nothing within cells. |

### 1. The formulas

- **What we did.** We wrote out the log-normal AFT model of Part 7 and the logistic model of Part 8, with their inputs and penalties.
- **What we found.** Both models suit the data. The AFT keeps the censored runs through their chance of lasting past the budget. The ridge penalty of the logistic model does not affect the AUC.
- **In the report.** Section 3.5 already has the AFT as Equations 10 and 11. The report plans to fit it with the settings only. The landmark version, with the kernel at step 2,000 next to the setting indicators, should be added there. The logistic model is not in the report yet and needs a sentence in Section 3.5.

### 2. Lookahead

- **What we did.** We checked that every input is read at step 2,000, before any run groks, and that every score holds out whole cells. We refitted the AFT with the kernel measured on the training pairs of the probe only.
- **What we found.** There is no lookahead. The kernel on the training pairs scores the same as the whole probe, with a pooled concordance of 0.901 both ways.
- **In the report.** Section 3.5 should state the landmark and that no run groks before step 2,983. Section 3.3 measures every statistic on the test pairs. Part 7 of `fitting.ipynb` finds that `A_t` on the 53 test pairs alone carries no signal, while `A_t` on the training pairs carries all of it. Section 3.3 should mention this.

### 3. Multicollinearity

- **What we did.** We measured the correlations between the three numbers across runs and within cells, the variance inflation factors, and the held-out AUC when one or two numbers are dropped.
- **What we found.** `A_t` and `S_t` are near twins across runs (r = 0.91, VIF up to 42), but not within cells (r of 0.30 or less). Their single coefficients are unstable, and a ranking by dropping one number at a time is noise.
- **In the report.** Section 3.5 should say that the three numbers are tested jointly. Section 4.2 should give the joint likelihood ratio test and the within-cell Cox coefficient of `A_t`, and not the single AFT coefficients.

### 4. Sample size

- **What we did.** We counted the grokked runs and the cells with a grokked run per coefficient in each set.
- **What we found.** The pooled set has 355 grokked runs and 75 grokked cells for 16 coefficients, which is enough. Weight decay 3e-5 is enough for when a run groks but not for whether. The level without weight decay has only 5 grokked cells for 10 coefficients.
- **In the report.** Section 3.6 (Choice of levels) argues for more runs at small weight decay, and this count supports it. Part 9 of `fitting.ipynb` lists which runs to resume or add. Section 5 should list the thin level without weight decay as a limit.

### 5. Weight decay

- **What we did.** We added kernel by weight decay interactions to the pooled AFT, fitted each level by itself, tested an `A_t` by weight decay interaction within cells, let the spread $\sigma$ differ by level, and held out one whole level at a time.
- **What we found.** The across-cell effect of the kernel differs by level (p = 1e-5 with the evidence cut by 5). The spread differs by level, from $\sigma$ = 0.19 at 3e-5 to 1.95 at 1e-3. The within-cell effect of `A_t` is the same at every level (p = 0.94). A model cannot place a level it has not seen: the concordance across levels drops from 0.88 and 0.90 to between 0.73 and 0.77.
- **In the report.** Section 3.5 plans to fit Equation 10 to the cells with weight decay with the single covariate log(ηλ) and one $\sigma$. If the spread also differs by level in that grid, one $\sigma$ will not fit, and lifelines can let $\sigma$ depend on the level with its `ancillary` argument. The broken line of Section 3.6 makes the same assumption. Section 4.2 should lead with the within-cell result. Section 5 should say that the model is not meant to predict an untested level.

### 6. The large L1 model

- **What we did.** We fitted an L1 logistic model to all 108 numbers logged at step 2,000, with the penalty picked in inner folds, and counted its picks over 100 redraws of the cells.
- **What we found.** It reaches a held-out AUC of 0.983 against 0.960. Its top picks track weight decay and the training accuracy, and it rarely picks `A_t`, `S_t` or `R_t`.
- **In the report.** It belongs in Appendix A as one sentence at most.

### 7. One number at a time

- **What we did.** We fitted a logistic model with one kernel number, alone and on top of the settings, and put ranges on the gaps with 1,000 redraws of the cells.
- **What we found.** Alone and pooled, `A_t` (AUC 0.868) beats `S_t` (0.763) and `R_t` (0.551), and every gap is clear. On top of the settings each number adds 0.015 or less, and they cannot be ranked.
- **In the report.** Section 4.2 can give one small table with both views. It replaces the earlier claim that `R_t` matters most.

### 8. Interactions

- **What we did.** We added the three products of the kernel numbers to the held-out models, tested them in the pooled AFT on top of squares and of the kernel by weight decay terms, and tested them within cells.
- **What we found.** They add a little when pooled (AUC 0.965 to 0.972) and nothing within cells (p = 0.83). They mostly stand in for curves and for the differences between weight decay levels. `A_t x S_t` is borderline.
- **In the report.** They should stay out of the main text. `A_t x S_t` could get one line in Appendix A.

### A caveat from Part 9 of `fitting.ipynb`

Part 9 pools every level and looks at `alpha`. Two of its results change how the checks above should be read.

- The held-out gain of the kernel in the pooled set (concordance 0.877 to 0.901) shrinks to 0.885 to 0.896 once the settings model has one indicator per combination of `alpha` and weight decay. Inside one value of `alpha`, the kernel adds almost nothing to the held-out scores. So about half of the pooled gain is the kernel standing in for combinations of the settings. The within-cell Cox result does not depend on this, and it holds at `alpha` 0.5 and 1.
- The kernel carries the effect of `alpha`. With `alpha` held out, the kernel alone reaches a concordance of 0.80, against 0.55 for width and weight decay. Section 5c above found that the kernel does not carry the effect of weight decay. This contrast bears on the lazy and rich account in Section 1 of the report, and it belongs in Section 4.2.

### What the report should say

1. **Formulas.** Write the AFT and the logistic model as in Section 1, and describe the AFT as a regression on log T that keeps the censored runs.
2. **Main result.** Lead with the within-cell Cox model. Among the seeds of one cell, a higher `A_t` at step 2,000 goes with earlier grokking, and that effect does not change with weight decay. This result is free of the overlap problem and of the weight decay problem.
3. **Prediction.** On held-out cells, adding the kernel at step 2,000 to the settings raises the pooled concordance from 0.877 to 0.901 and the pooled AUC from 0.950 to 0.965. Report the joint likelihood ratio test for the three numbers together, and say that about half of the gain goes away when the settings may interact.
4. **Weight decay.** Give each level its own term. Say that it shifts the grokking time and changes its spread, and that the across-cell link between the kernel and the time differs by level. Do not claim that the model predicts an untested level.
5. **`alpha`.** Say that the kernel at step 2,000 carries the effect of `alpha`, which it does not do for weight decay.
6. **Which number matters.** Use the one-number-at-a-time table of Section 7. Alone, `A_t` is the most informative. On top of the settings, the three add a similar small amount.
7. **Things to leave out.** These are the claim that `R_t` matters most, any reading of the sign of `S_t` on its own, the level without weight decay as a headline, a section on the L1 model, and kernel by kernel interactions in the main text.
8. **Limits.** The AFT intervals treat the seeds as independent, so they are too narrow. Weight decay 3e-5 was chosen after looking at the outcomes. The results use one landmark, step 2,000. These belong in Section 5.